# ИДЗ 3. Сравнение регрессионных моделей

 строятся четыре вида регрессии:
линейная, степенная, показательная и гиперболическая.



## Исходные данные и план работы



In [78]:
import numpy as np

x_values = np.array([45.1, 59.0, 57.2, 61.8, 58.8, 47.2, 55.2], dtype=float)
y_values = np.array([68.8, 61.2, 59.9, 56.7, 55.0, 54.3, 49.3], dtype=float)


## Что будем считать

Во всех моделях используем одни и те же показатели качества:

$$
A = \frac{100}{n} \sum_{i=1}^{n} \left|
    \frac{y_i - \hat{y_i }}{y_i} \right|
$$

$$
F_{	ext{эмп}} = \frac{r^2}{1-r^2}(n-2)
$$

Если `F_эмп <= F_кр`, то статистическая значимость не подтверждается.
Для этой выборки `n = 7`, поэтому критическое значение по таблице Фишера
при уровне значимости `0.05` равно `F_кр = 6.61`.

В преобразованных моделях коэффициент корреляции считаем уже для
линейризованных данных, а не для исходных `x_values` и `y_values`.


In [79]:
F_CRIT = 6.61  # Табличное значение F(0.95; 1, 5) для n = 7

def fit_line(x_vals, y_vals):
    x_mean = np.mean(x_vals)
    y_mean = np.mean(y_vals)
    x_var = np.var(x_vals)
    xy_mean = np.mean(x_vals * y_vals)
    slope = (xy_mean - x_mean * y_mean) / x_var
    intercept = y_mean - slope * x_mean
    return float(intercept), float(slope)

def evaluate_model(y_true, y_pred, corr):
    n = len(y_true)
    avg_rel_error = float(np.mean(np.abs((y_true - y_pred) / y_true)) * 100)
    fisher_emp = float(corr ** 2 / (1 - corr ** 2) * (n - 2))
    return avg_rel_error, fisher_emp

def format_linear_equation(a, b):
    sign = '+' if b >= 0 else '-'
    return f'y = {a:.2f} {sign} {abs(b):.2f}x'

def format_power_equation(a, b):
    return f'y = {a:.2f} * x^{b:.2f}'

def format_exponential_equation(a, b):
    return f'y = {a:.2f} * {b:.2f}^x'

def format_hyperbola_equation(a, b):
    sign = '+' if b >= 0 else '-'
    return f'y = {a:.2f} {sign} {abs(b):.2f} / x'

results = []


## 1. Линейная регрессия



$$
y = a + bx
$$


$$
b = \frac{\overline{xy} - \overline{x}\,\overline{y}}{\sigma_x^2}, \qquad
a = \overline{y} - b\,\overline{x}
$$


In [80]:
linear_intercept, linear_slope = fit_line(x_values, y_values)
linear_pred = linear_intercept + linear_slope * x_values
linear_corr = float(np.corrcoef(x_values, y_values)[0, 1])
linear_A, linear_F = evaluate_model(y_values, linear_pred, linear_corr)

print('Линейная регрессия')
print(format_linear_equation(linear_intercept, linear_slope))
print(f'r = {linear_corr:.2f}')
print(f'A = {linear_A:.2f} %')
print(f'F_эмп = {linear_F:.2f}')
print(f'F_кр = {F_CRIT:.2f}')
print('Вывод: статистическая значимость на уровне 5% не подтверждается')
print()

results.append(('Линейная', linear_A, linear_F))


Линейная регрессия
y = 76.88 - 0.35x
r = -0.35
A = 8.14 %
F_эмп = 0.71
F_кр = 6.61
Вывод: статистическая значимость на уровне 5% не подтверждается



## 2. Степенная регрессия



$$
y = ax^b
$$


$$
\log_{10} y = \log_{10} a + b\,\log_{10} x
$$




In [81]:
log_x_values = np.log10(x_values)
log_y_values = np.log10(y_values)
power_log_intercept, power_exponent = fit_line(log_x_values, log_y_values)
power_coeff = float(10 ** power_log_intercept)
power_pred = power_coeff * (x_values ** power_exponent)
power_corr = float(np.corrcoef(log_x_values, log_y_values)[0, 1])
power_A, power_F = evaluate_model(y_values, power_pred, power_corr)

print('Степенная регрессия')
print(format_power_equation(power_coeff, power_exponent))
print(f'r = {power_corr:.2f}')
print(f'A = {power_A:.2f} %')
print(f'F_эмп = {power_F:.2f}')
print(f'F_кр = {F_CRIT:.2f}')
print('Вывод: статистическая значимость на уровне 5% не подтверждается')
print()

results.append(('Степенная', power_A, power_F))


Степенная регрессия
y = 190.03 * x^-0.30
r = -0.34
A = 8.05 %
F_эмп = 0.65
F_кр = 6.61
Вывод: статистическая значимость на уровне 5% не подтверждается



## 3. Показательная регрессия



$$
y = ab^x
$$


$$
\log_{10} y = \log_{10} a + x\,\log_{10} b
$$



In [82]:
exp_log_intercept, exp_log_slope = fit_line(x_values, log_y_values)
exp_coeff = float(10 ** exp_log_intercept)
exp_base = float(10 ** exp_log_slope)
exp_pred = exp_coeff * (exp_base ** x_values)
exp_corr = float(np.corrcoef(x_values, log_y_values)[0, 1])
exp_A, exp_F = evaluate_model(y_values, exp_pred, exp_corr)

print('Показательная регрессия')
print(format_exponential_equation(exp_coeff, exp_base))
print(f'r = {exp_corr:.2f}')
print(f'A = {exp_A:.2f} %')
print(f'F_эмп = {exp_F:.2f}')
print(f'F_кр = {F_CRIT:.2f}')
print('Вывод: статистическая значимость на уровне 5% не подтверждается')
print()

results.append(('Показательная', exp_A, exp_F))


Показательная регрессия
y = 77.24 * 0.99^x
r = -0.32
A = 8.08 %
F_эмп = 0.57
F_кр = 6.61
Вывод: статистическая значимость на уровне 5% не подтверждается



## 4. Гиперболическая регрессия



$$
y = a + \frac{b}{x} = a + bz.
$$



In [83]:
inverse_x_values = 1 / x_values
hyper_intercept, hyper_slope = fit_line(inverse_x_values, y_values)
hyper_pred = hyper_intercept + hyper_slope * inverse_x_values
hyper_corr = float(np.corrcoef(inverse_x_values, y_values)[0, 1])
hyper_A, hyper_F = evaluate_model(y_values, hyper_pred, hyper_corr)

print('Гиперболическая регрессия')
print(format_hyperbola_equation(hyper_intercept, hyper_slope))
print(f'r = {hyper_corr:.2f}')
print(f'A = {hyper_A:.2f} %')
print(f'F_эмп = {hyper_F:.2f}')
print(f'F_кр = {F_CRIT:.2f}')
print('Вывод: статистическая значимость на уровне 5% не подтверждается')
print()

results.append(('Гиперболическая', hyper_A, hyper_F))


Гиперболическая регрессия
y = 38.44 + 1054.67 / x
r = 0.39
A = 8.06 %
F_эмп = 0.91
F_кр = 6.61
Вывод: статистическая значимость на уровне 5% не подтверждается



#


In [84]:
print('Сравнение моделей по средней относительной ошибке:')
print(f"{'Модель':<16} {'A, %':>8} {'F_эмп':>8}")
print('-' * 34)
for name, error, fisher in results:
    print(f'{name:<16} {error:8.2f} {fisher:8.2f}')
print()
best_name, best_A, _ = min(results, key=lambda item: item[1])
print(f'Минимальная ошибка у модели: {best_name} ({best_A:.2f} %)')
print('Но разница между моделями небольшая, а критерий Фишера ни одна из них не проходит.')


Сравнение моделей по средней относительной ошибке:
Модель               A, %    F_эмп
----------------------------------
Линейная             8.14     0.71
Степенная            8.05     0.65
Показательная        8.08     0.57
Гиперболическая      8.06     0.91

Минимальная ошибка у модели: Степенная (8.05 %)
Но разница между моделями небольшая, а критерий Фишера ни одна из них не проходит.
